# Chapter 4.1-4.2: Text Features and Prompt Engineering

Goal: Understand why ML needs numeric features from text, practice traditional text extraction methods, then learn to write effective LLM prompts for structured data extraction.

### Topics:
- Why ML models can't consume raw text directly
- Keyword matching for sentiment analysis
- Writing effective LLM prompts (good vs bad)
- Requesting JSON output format from LLMs
- Extracting main topics and sub-topics from text
- Aspect-based sentiment (sentiment per sub-topic)
- Few-shot prompting to improve extraction quality

In [1]:
import pandas as pd
import numpy as np
import json
from openai import OpenAI

## API key setup

Paste the temporary OpenAI API key your professor provides into the cell below. This key is only for use during this in-class activity.

**Important:**
- Do not commit this notebook to GitHub with the key filled in.
- Delete the key from the cell before saving/pushing your work.
- The key will be rotated after class, so don't plan on reusing it later.

In [ ]:
# Paste the temporary API key here (the one the professor gave out in class).
# Do NOT commit this notebook to GitHub with the key filled in.
# Delete the key from this cell after class.
OPENAI_API_KEY = ""

client = OpenAI(api_key=OPENAI_API_KEY)

## Quick Recap

- **Feature engineering**: The process of creating numeric inputs for ML models from raw data
- **Text features**: Numeric representations of text data (word counts, sentiment scores, extracted fields)
- **Keyword matching**: Counting occurrences of known words to estimate properties like sentiment
- **Prompt engineering**: Crafting LLM instructions to get consistent, structured outputs
- **Main topic / sub-topic extraction**: Asking the LLM to identify what a piece of text is *about*
- **Aspect-based sentiment**: Getting a separate sentiment label for each sub-topic in the same review
- **Few-shot prompting**: Including examples in the prompt to guide LLM behavior

## Data

We'll work with product reviews for electronics (coffee makers, headphones, chargers). All data is defined inline — no files to download.

In Part 2 we'll send these reviews to OpenAI's API to extract sentiment, main topics, sub-topics, and aspect-level sentiment.

In [3]:
reviews = [
    {"id": 1, "product": "coffee maker", "text": "Absolutely love this coffee maker! Best purchase I've made all year. Brews perfect coffee every morning for $49.99."},
    {"id": 2, "product": "headphones", "text": "Terrible sound quality. Broke after two weeks. Complete waste of $79.99. Do NOT buy these."},
    {"id": 3, "product": "charger", "text": "It charges my phone. Nothing special but it works. Paid $12.99 which seems fair."},
    {"id": 4, "product": "coffee maker", "text": "The coffee tastes burnt and the machine is loud. Returned it after 3 days. Worst $89.99 I've ever spent."},
    {"id": 5, "product": "headphones", "text": "Great noise cancellation and comfortable fit. Battery lasts forever. Worth every penny of the $199.99."},
    {"id": 6, "product": "charger", "text": "Fast charging is nice but the cable feels cheap. Worried it won't last. $24.99 is okay I guess."},
    {"id": 7, "product": "coffee maker", "text": "Does the job but nothing more. Average coffee, average build quality. It's fine for $39.99."},
    {"id": 8, "product": "headphones", "text": "I wanted to love these SO much but the Bluetooth keeps disconnecting. Sound is amazing when it works though. $149.99 feels like a gamble."},
    {"id": 9, "product": "charger", "text": "Bought 3 of these at $9.99 each. Two work great, one was dead on arrival. Hit or miss quality."},
    {"id": 10, "product": "coffee maker", "text": "Upgraded from my old drip machine and WOW. The difference is night and day. $129.99 well spent."},
    {"id": 11, "product": "headphones", "text": "Sure, they work. If you like mediocre sound and uncomfortable ear cups. Save your money."},
    {"id": 12, "product": "charger", "text": "This charger literally saved my phone on a road trip. Charges super fast. Best $19.99 ever."},
    {"id": 13, "product": "coffee maker", "text": "Oh great, another coffee maker that leaks everywhere. Just what I needed. Thanks for nothing."},
    {"id": 14, "product": "headphones", "text": "Not bad for the price. Sound is decent, build quality is acceptable. Would buy again at $59.99."},
    {"id": 15, "product": "charger", "text": "Works exactly as described. Fast, reliable, good build. My third one from this brand. $14.99 is a steal."}
]

## Part 1: Traditional Approaches

Before LLMs existed, data scientists had to extract features from text using rule-based methods. These are still useful — they're fast, free, and predictable. Let's see what they can (and can't) do.

### 1. By hand — Manually label sentiment

Read the first 10 reviews and assign each a sentiment label: `"positive"`, `"negative"`, or `"mixed"`. This is your **ground truth** — the labels you'll compare automated methods against.

In [ ]:
# Print the first 10 reviews so you can read them
for r in reviews[:10]:
    print(f"Review {r['id']} ({r['product']}): {r['text']}")
    print()

In [ ]:
# Assign your manual labels here
manual_labels = {
    1: ...,  # "positive", "negative", or "mixed"
    2: ...,
    3: ...,
    4: ...,
    5: ...,
    6: ...,
    7: ...,
    8: ...,
    9: ...,
    10: ...,
}

### 2. By hand — Pick keywords for a sentiment classifier

The function `simple_sentiment(text)` below is already written for you. It counts how many positive and negative words appear in the review and returns a label based on which count is higher.

**Your job:** fill in the `positive_words` and `negative_words` lists with words you think signal positive and negative sentiment. Look at the reviews above for inspiration. Then run the comparison against your manual labels from Exercise 1 and see how well your keyword list performs.

In [ ]:
# Fill in these two lists with keywords you think signal positive/negative sentiment.
positive_words = [
    ...,
]

negative_words = [
    ...,
]

def simple_sentiment(text):
    """Classify sentiment based on keyword counts."""
    text_lower = text.lower()
    pos_count = sum(1 for w in positive_words if w in text_lower)
    neg_count = sum(1 for w in negative_words if w in text_lower)

    if pos_count > neg_count:
        return "positive"
    elif neg_count > pos_count:
        return "negative"
    else:
        return "mixed"


In [ ]:
# Compare keyword-based results to your manual labels
print("Review | Manual Label | Keyword Label | Match?")
print("-" * 50)
for r in reviews[:10]:
    keyword_label = simple_sentiment(r["text"])
    manual = manual_labels[r["id"]]
    match = "Yes" if keyword_label == manual else "No"
    print(f"  {r['id']:2d}    | {manual:10s} | {keyword_label:13s} | {match}")

**Your observation:** Which reviews did the keyword approach get wrong? Why did it fail? Are there reviews where *no* reasonable keyword list could get the right answer?

(Write your answer here)

## Part 2: Prompt Engineering

Keyword matching is fast and free, but it misses nuance. LLMs can understand context, sarcasm, and complex meaning — but only if you prompt them well. Let's practice writing effective prompts.

For the rest of the activity we'll be making real calls to the OpenAI API. Each call costs a tiny amount (fractions of a cent), so feel free to experiment.

In [22]:
# Small helper so we don't repeat the API call boilerplate in every exercise.
def call_llm(prompt, max_tokens=200):
    response = client.chat.completions.create(
        model="gpt-5-nano-2025-08-07",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

This function is just a wrapper that helps us ask ChatGPT questions. This is no different from going to chatgpt.com and typing in a question.

In [ ]:
call_llm("What's the date today?")

### 3. See a bad prompt in action

Here's a vague, poorly-written sentiment-analysis prompt. Run it on a few reviews and look at what comes back.

In [ ]:
bad_prompt = """What do you think about this review?

Review: {review_text}"""

# Grab just the first review
r = reviews[0]

# Format the prompt
prompt = bad_prompt.format(review_text=r["text"])

# Print the prompt
print(f'Prompt to send to the LLM: {prompt}')
print()

# Send it to the LLM for feedback
print('- Response: -')
response = call_llm(prompt)
print(response)
print()

**What's wrong with the responses above?** Think about what you'd do if you had to feed these into a pandas DataFrame. Every response is a different length, a different format, and doesn't give you a single clean label.

(Write your answer here)

### 4. By hand — Write a better prompt

Now it's your turn. Write a prompt that gets the LLM to return *only* a sentiment label and nothing else — no explanations, no extra words, no punctuation beyond the label itself.

Your prompt should:
- Clearly state the task (classifying the sentiment of a product review)
- Specify the exact allowed outputs: `positive`, `negative`, or `mixed`
- Tell the model to respond with only that one word
- Include a `{review_text}` placeholder so we can format it with any review

Then run it on the same 3 reviews (1, 8, 13) and compare the output to the bad prompt above.

In [ ]:
# Write your improved prompt here. Replace the ... with your prompt text.
good_prompt = """
...
"""

# Run your prompt through the LLM on the first 10 reviews and build a comparison
# table showing manual label, keyword label, and LLM label side-by-side.
rows = []
for r in reviews[:5]:
    llm_label = call_llm(good_prompt.format(review_text=r["text"]), max_tokens=10)
    keyword_label = simple_sentiment(r["text"])
    manual = manual_labels[r["id"]]
    rows.append({
        "review_id": r["id"],
        "manual": manual,
        "keyword": keyword_label,
        "llm": llm_label,
        "all_agree": manual == keyword_label == llm_label,
    })

pd.DataFrame(rows)

**Checklist — does your prompt produce output that:**
- [ ] Is exactly one word?
- [ ] Is always one of `positive`, `negative`, or `mixed`?
- [ ] Could be dropped straight into a pandas DataFrame column without any cleaning?

If any of these fail, tweak your prompt and try again. This is prompt engineering — iterate until it works.

**Your analysis:** Where do the three methods agree? Where do they disagree? Which reviews does the LLM handle better than the keyword approach? Are there any reviews where the LLM gets it wrong?

(Write your answer here)

### 5. By hand — Level up: ask for JSON with confidence and reasoning

A single label is great for quick classification, but often you want more. Write a new prompt that returns **JSON** with three fields:
- `sentiment`: one of `"positive"`, `"negative"`, `"mixed"`
- `confidence`: a float between 0 and 1
- `reasoning`: a short string (one sentence) explaining the classification

Your prompt should tell the model to respond with *only* the JSON — no markdown code fences, no extra text. Then run it through `json.loads()` to verify it actually parses. We'll run it on the first 10 reviews and show the results in a DataFrame.

In [ ]:
# Write your JSON-returning prompt here.
json_prompt = """
...
"""

rows = []
for r in reviews[:5]:
    response = call_llm(json_prompt.format(review_text=r["text"]), max_tokens=150)
    parsed = json.loads(response)
    rows.append({
        "review_id": r["id"],
        "sentiment": parsed["sentiment"],
        "confidence": parsed["confidence"],
        "reasoning": parsed["reasoning"],
    })

pd.DataFrame(rows)

### 6. By hand — Extract main topic and sub-topics

Sentiment is only one piece of text analysis. Another really common task is figuring out *what a piece of text is about*.

**Your job:** write a prompt that, for a given review, returns a JSON object with exactly these two fields:
- `main_topic`: a single string, one of `"product_quality"`, `"price_value"`, `"shipping"`, `"customer_service"`, or `"other"`
- `sub_topics`: a list of 1-3 short strings describing the specific aspects the reviewer discussed (e.g., `["battery life", "sound quality"]`)

The response must be *only* the JSON — no extra text or markdown fences — so that `json.loads()` can parse it directly. Include a `{review_text}` placeholder in your prompt.

In [ ]:
# Write your prompt here.
topic_prompt = """
...
"""

# Skeleton below runs your prompt on the first 10 reviews and builds a DataFrame.
rows = []
for r in reviews[:5]:
    response = call_llm(topic_prompt.format(review_text=r["text"]), max_tokens=150)
    parsed = json.loads(response)
    rows.append({
        "review_id": r["id"],
        "main_topic": parsed["main_topic"],
        "sub_topics": parsed["sub_topics"],
    })

pd.DataFrame(rows)

### 7. By hand — Aspect-based sentiment (sentiment per sub-topic)

A single "positive/negative/mixed" label for a whole review throws away a lot of information. Review 8 is a great example — the reviewer *loves* the sound quality but *hates* the Bluetooth. That's two different opinions about two different aspects of the same product.

**Your job:** write a prompt that returns a JSON object with a single field `aspects`, whose value is a list of objects each containing:
- `topic`: a short string describing the sub-topic (e.g., `"sound quality"`)
- `sentiment`: one of `"positive"`, `"negative"`, or `"mixed"`

Include every distinct aspect discussed in the review (usually 1-4). Respond with *only* the JSON so `json.loads()` parses it directly. Include a `{review_text}` placeholder. We'll run it on the first 10 reviews.

In [ ]:
# Write your prompt here.
aspect_prompt = """
...
"""

# Skeleton below runs your prompt on the first 10 reviews and flattens the
# aspects into a single DataFrame with one row per (review, aspect) pair.
rows = []
for r in reviews[:5]:
    response = call_llm(aspect_prompt.format(review_text=r["text"]), max_tokens=250)
    parsed = json.loads(response)
    for aspect in parsed["aspects"]:
        rows.append({
            "review_id": r["id"],
            "topic": aspect["topic"],
            "sentiment": aspect["sentiment"],
        })

pd.DataFrame(rows)

### 8. Use AI — Build a few-shot prompt function

Use your AI assistant to write a function `build_few_shot_prompt(text, examples, task)` that:
1. Takes a review text, a list of example (text, label) pairs, and a task description
2. Builds a prompt that includes the examples before the actual review
3. Returns the complete prompt string

Use 3 examples: one positive, one negative, one mixed.

In [ ]:
# Define 3 examples for few-shot prompting
sentiment_examples = [
    ("This product is fantastic! Works perfectly every time.",
     '{"sentiment": "positive", "confidence": 0.95, "reasoning": "Strong positive language with no complaints."}'),
    ("Broke on day one. Terrible quality. Want my money back.",
     '{"sentiment": "negative", "confidence": 0.97, "reasoning": "Product failure, strong negative language, refund request."}'),
    ("Good features but the battery dies too fast. Hard to recommend.",
     '{"sentiment": "mixed", "confidence": 0.80, "reasoning": "Acknowledges positives but significant negative outweighs them."}'),
]

# Use AI to write this function
def build_few_shot_prompt(text, examples, task):
    """Build a few-shot prompt with examples.

    Args:
        text: The review to analyze
        examples: List of (input_text, expected_output) tuples
        task: Description of the task

    Returns:
        Complete prompt string with examples
    """
    ...

# Test it
test_prompt = build_few_shot_prompt(
    reviews[7]["text"],  # Review 8 — the tricky mixed one
    sentiment_examples,
    "Analyze the sentiment of the following product review. Return JSON with sentiment, confidence, and reasoning."
)
print(test_prompt)

### 9. By hand — Compare zero-shot vs few-shot on the first 10 reviews

Let's compare how zero-shot and few-shot prompts handle all 10 reviews. Some of them are tricky — review 8 is mixed ("I wanted to love these SO much but the Bluetooth keeps disconnecting...") and other reviews use sarcasm or understatement.

We'll run each review twice: once with just the task description (zero-shot), and once with the 3 labeled examples you defined above (few-shot).

**Your job:** write the `sentiment_task` description below. It should describe the sentiment classification task in plain English and specify that the LLM should respond with JSON containing `sentiment`, `confidence`, and `reasoning`. This string will be fed into both the zero-shot and few-shot prompts so we can see the effect of the examples alone.

In [ ]:
# Write the task description string that will be used in both zero-shot and
# few-shot prompts. It should describe what the LLM needs to do and what the
# output should look like (JSON with sentiment, confidence, reasoning).
sentiment_task = (
    ...
)

def zero_shot_prompt(text, task):
    return f"{task}\n\nReview: {text}"

rows = []
for r in reviews[:5]:
    zero_response = call_llm(zero_shot_prompt(r["text"], sentiment_task), max_tokens=200)
    few_response = call_llm(
        build_few_shot_prompt(r["text"], sentiment_examples, sentiment_task),
        max_tokens=200,
    )

    zero_parsed = json.loads(zero_response)
    few_parsed = json.loads(few_response)

    rows.append({
        "review_id": r["id"],
        "manual": manual_labels[r["id"]],
        "zero_shot": zero_parsed["sentiment"],
        "few_shot": few_parsed["sentiment"],
        "agree": zero_parsed["sentiment"] == few_parsed["sentiment"],
    })

pd.DataFrame(rows)

**Your analysis:**

1. On which reviews do zero-shot and few-shot disagree? Which one matches your manual labels more often?

(Write your answer here)

2. Look at the reviews with sarcasm or mixed sentiment. Did few-shot handle them better than zero-shot? Why do you think that is?

(Write your answer here)

3. What kind of examples would you include in a few-shot prompt to handle sarcasm better?

(Write your answer here)

## Discussion

1. What text information can't be captured by keyword matching that an LLM could extract?
2. Aspect-based sentiment gave us a richer picture of review 8 than a single label ever could. What else might an LLM be able to pull out of these reviews that a keyword count couldn't?
3. What are the tradeoffs of using an LLM vs. a traditional method? Think about cost, consistency, speed, and interpretability. (You can check `response.usage` on any API response to see how many tokens each call used.)

(Discuss with a neighbor)